### GPU Envornment

In [14]:
from pathlib import Path
import csv
import sys
import torch
from ultralytics import YOLO
from IPython.display import display, Video

PROJECT = Path.home() / 'projects' / 'YOLO'
MODEL_PATH = PROJECT / 'yolo26n.pt'

# Video to test on KSU DGX
VIDEO_PATH = PROJECT / 'data' / 'videos' / 'CummingCars_2.mov'

OUTPUT_DIR = PROJECT / 'runs' / 'notebook_predictions' / 'yolo26n'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('Python:', sys.executable)
print('CUDA available:', torch.cuda.is_available())
print('Prediction device:', DEVICE)
print('Model:', MODEL_PATH)
print('Video:', VIDEO_PATH)

if not VIDEO_PATH.is_file():
    print('Video not found. Check VIDEO_PATH.')

Python: /mnt/bst/dgx/atomitak/aadhika2/venvs/yolo-basic/bin/python
CUDA available: True
Prediction device: 0
Model: /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/yolo26n.pt
Video: /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov


### Load model and run prediction

In [15]:
model = YOLO(str(MODEL_PATH))

results = model.predict(
    source=str(VIDEO_PATH),
    device=DEVICE,
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(OUTPUT_DIR),
    name=VIDEO_PATH.stem,
    exist_ok=True,
    stream=True
)

for result in results:
    pass

print("Video processing completed.")
print("Results saved in:", OUTPUT_DIR / VIDEO_PATH.stem)


video 1/1 (frame 1/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 1 car, 1 truck, 11.6ms
video 1/1 (frame 2/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 1 car, 1 truck, 12.8ms
video 1/1 (frame 3/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 4 cars, 12.3ms
video 1/1 (frame 4/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 5 cars, 14.2ms
video 1/1 (frame 5/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 3 cars, 12.5ms
video 1/1 (frame 6/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 3 cars, 14.6ms
video 1/1 (frame 7/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 4 cars, 12.2ms
video 1/1 (frame 8/468) /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/data/videos/CummingCars_2.mov: 384x640 2 cars, 1 truc

### Show bounding boxes and save the annotated image

In [16]:
from IPython.display import display, Video

# Output folder for this video
video_output_dir = OUTPUT_DIR / VIDEO_PATH.stem

# Find the annotated video saved by YOLO
saved_videos = (
    list(video_output_dir.glob("*.avi")) +
    list(video_output_dir.glob("*.mp4")) +
    list(video_output_dir.glob("*.mov"))
)

if saved_videos:
    out_video = saved_videos[0]
    print("Video saved:", out_video)
    print("File size:", round(out_video.stat().st_size / (1024 * 1024), 2), "MB")
else:
    print("No output video found. Run the prediction cell first.")

Video saved: /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/runs/notebook_predictions/yolo26n/CummingCars_2/CummingCars_2.avi
File size: 150.67 MB


### Save class names, confidence scores, and box coordinates

In [17]:
from ultralytics import YOLO
from pathlib import Path
import csv

model = YOLO(str(MODEL_PATH))

# Create output directory
video_output_dir = OUTPUT_DIR / VIDEO_PATH.stem
video_output_dir.mkdir(parents=True, exist_ok=True)

csv_path = video_output_dir / f"{VIDEO_PATH.stem}_yolo26n_boxes.csv"

columns = [
    'video',
    'frame',
    'class_name',
    'class_id',
    'confidence',
    'x1', 'y1', 'x2', 'y2'
]

# Run prediction on the video
results = model.predict(
    source=str(VIDEO_PATH),
    device=DEVICE,
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(OUTPUT_DIR),
    name=VIDEO_PATH.stem,
    exist_ok=True,
    stream=True,
    verbose=False
)

frame_count = 0
detection_count = 0

with csv_path.open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=columns)
    writer.writeheader()

    for frame_num, result in enumerate(results, start=1):
        frame_count = frame_num

        if result.boxes is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.int().cpu().numpy()

        for cls_id, conf, box in zip(classes, confs, boxes):
            writer.writerow({
                'video': VIDEO_PATH.name,
                'frame': frame_num,
                'class_name': result.names[int(cls_id)],
                'class_id': int(cls_id),
                'confidence': round(float(conf), 5),
                'x1': round(float(box[0]), 2),
                'y1': round(float(box[1]), 2),
                'x2': round(float(box[2]), 2),
                'y2': round(float(box[3]), 2),
            })

            detection_count += 1

print("Video processing completed")
print("Total frames:", frame_count)
print("Total detections:", detection_count)
print("CSV saved:", csv_path)
print("Output folder:", video_output_dir)

Results saved to /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/runs/notebook_predictions/yolo26n/CummingCars_2
Video processing completed
Total frames: 468
Total detections: 1059
CSV saved: /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/runs/notebook_predictions/yolo26n/CummingCars_2/CummingCars_2_yolo26n_boxes.csv
Output folder: /mnt/bst/dgx/atomitak/aadhika2/projects/YOLO/runs/notebook_predictions/yolo26n/CummingCars_2


In [18]:
from pathlib import Path

video_dir = OUTPUT_DIR / VIDEO_PATH.stem

for f in video_dir.iterdir():
    print(f.name, f.stat().st_size / (1024 * 1024), "MB")

CummingCars_2.avi 150.6721248626709 MB
CummingCars_2_yolo26n_boxes.csv 0.06659221649169922 MB


In [13]:
which ffmpeg

SyntaxError: invalid syntax (1257492167.py, line 1)